In [ ]:
import math_toolkit
math_toolkit.notebook.activate()

# plot

`plot(...)` samples one symbolic expression as a real curve in a live figure, with optional parameters, labels, styles, and a reusable plot handle.

Preferred forms:

```python
fig = figure("demo")
fig.show()
fig.plot(expr, (x, xmin, xmax))
```

```python
fig = figure("demo")
fig.show()
with fig:
    plot(expr, x)
```

In [ ]:
x, a = symbols("x a")

fig = figure("plot-quick-payoff")
fig.show()
fig.parameters({a: 3.0})
fig.plot(exp(-x**2) * cos(a * x), (x, -4, 4), label="damped cosine")

## Core behavior

`fig.plot(expr, domain)` adds or updates one real scalar curve $y=f(x)$ in a figure. The domain may be a symbol such as `x` or a finite interval tuple such as `(x, -10, 10)`.

The top-level `plot(expr, domain)` command has the same plotting behavior, but reusable documentation should make its target explicit with `fig.plot(...)` or `with fig:`.

Plot commands return a `PlotHandle`. The handle knows its parent figure and can display the figure, change the label, merge style updates, change parameter slider values, or remove the plot.

In [ ]:
fig = figure("plot-handle-basics")
fig.show()

handle = fig.plot(sin(x), (x, -pi, pi), name="wave", label="original")
handle.label = "sin x"
handle.style.color = "crimson"
handle.style.width = 3

md("""The named plot handle is called `{{ handle.name }}` and displays as `{{ handle.label }}`.""")

## Common patterns

**Line curves.** Use `fig.plot(expr, x)` for the default visible window, or `fig.plot(expr, (x, xmin, xmax))` for a finite sampled interval.

**Other plot kinds.** Use [list_plot](list_plot.ipynb) for discrete point sets, [temperature_plot](temperature_plot.ipynb) for heatmaps, [contour_plot](contour_plot.ipynb) for level curves, [domain_plot](domain_plot.ipynb) for Boolean or signed regions, and [parametric_plot](parametric_plot.ipynb) for two-coordinate curves.

**Plot names.** Use `name=` when a later call should update or retrieve the same plot. Use `label=` for the display label; it does not define identity.

In [ ]:
x, y, t = symbols("x y t")

field_fig = figure("plot-kinds")
field_fig.show()
field_fig.plot(sin(x), (x, -pi, pi), name="sine")
field_fig.plot(cos(x), (x, -pi, pi), name="cosine")

## Options and Details

**Parameters.** `fig.parameters(...)` creates sliders for symbols that are not the plotted domain variables. The compact form is `{symbol: value}`. Dictionary specs can also supply slider metadata such as bounds, step, value, and label when finer control is needed.

**Samples.** `samples=` controls sampling density. Curves and parametric curves accept one integer. Two-dimensional plot kinds accept either one integer for both axes or `(x_samples, y_samples)`.

**Line styles.** `fig.plot(...)` supports style keys such as `color`, `width`, `opacity`, `visible`, and `dash`.

**Per-call figure routing.** Top-level plot commands accept `figure=` as a figure name or figure handle. Prefer the figure method form when writing examples because the target is already explicit.

In [ ]:
x, amplitude, frequency = symbols("x amplitude frequency")

fig = figure("plot-params-style")
fig.show()
fig.parameters({
    amplitude: {"value": 1.0, "min": 0.2, "max": 2.0, "step": 0.1, "label": "amplitude"},
    frequency: {"value": 1.0, "min": 0.5, "max": 5.0, "step": 0.25, "label": "frequency"},
})
fig.plot(
    amplitude * sin(frequency * x),
    (x, -2 * pi, 2 * pi),
    style={"color": "royalblue", "width": 3},
    samples=500,
)

## Plot notes

Use `fig.info(...)` when a curve needs a live note beside the legend and parameter sliders. Info expressions share figure parameters with plots, so one slider can control both the sampled curve and the displayed scalar note.

In [ ]:
note_fig = figure("plot-with-info")
note_fig.show()
note_fig.parameters({a: {"value": 2.0, "min": 0.5, "max": 3.0}})
note_fig.plot(a * exp(-x**2), (x, -3, 3), name="bell")
note_fig.info("Amplitude: ", a, name="amplitude-note", title="Curve note")

## Semantics

Plotting crosses from symbolic expressions into numerical sampling. For `fig.plot(...)`, the plotted expression must become one real scalar value at each sampled domain point.

Domain variables are explicit. `fig.plot(expr, x)` means sample `expr` as a function of `x`; `fig.plot(expr, (x, -1, 1))` means use the finite interval from `-1` to `1`. Symbols not used as domain variables can become parameters with `fig.parameters(...)`.

A plot `name=` is an identity inside one figure. Calling the same plot kind with the same `name=` updates that plot in place. `label=` is only the displayed legend/control label.

In [ ]:
fig = figure("named-plot-update")
fig.show()

fig.plot(sin(x), (x, -pi, pi), name="curve", label="first curve")
updated = fig.plot(cos(x), name="curve", label="updated curve")

md("""The plot named `{{ updated.name }}` now displays as `{{ updated.label }}`.""")

## Examples and Applications

### Example: compare a function with a local approximation

Several scalar curves can share one figure when the comparison is the point.

In [ ]:
fig = figure("curve-comparison")
fig.show()

fig.plot(sin(x), (x, -pi, pi), name="sine", label="sin x")
fig.plot(x - x**3 / 6, (x, -pi, pi), name="cubic", label="cubic approximation", style={"dash": "dash"})

## Pitfalls

**Do not omit the plot variable.** `fig.plot(sin(x))` is ambiguous; write `fig.plot(sin(x), x)` or `fig.plot(sin(x), (x, -10, 10))`.

**Use symbolic comparisons for domains.** `fig.domain_plot(x > 0, x, y)` is symbolic. A plain Python `True` or `False` is not a plotted mathematical condition.

**Use the right plot kind for the shape.** `fig.plot(...)` is for continuous scalar curves. Use `fig.list_plot(...)` for finite point sets, `fig.parametric_plot(...)` for two-coordinate curves, and `fig.temperature_plot(...)` or `fig.contour_plot(...)` for scalar fields.

**Held or nonnumeric expressions may need preparation.** If an expression cannot be compiled numerically, simplify it, substitute required values, or unhold held definitions before plotting.

**Updating by name keeps the plot kind contract.** A named update still has to be valid for the plot kind you are calling.

## See also

- [figure](figure.ipynb) for durable plotting containers, display, context routing, layouts, and views.
- [list_plot](list_plot.ipynb) for discrete point sets.
- [temperature_plot](temperature_plot.ipynb) for heatmap scalar fields.
- [contour_plot](contour_plot.ipynb) for contour scalar fields.
- [domain_plot](domain_plot.ipynb) for Boolean and signed domains.
- [parametric_plot](parametric_plot.ipynb) for two-coordinate parametric curves.
- [Num](Num.ipynb) for compiling symbolic expressions to numeric callables.
- [Expression](Expression.ipynb) for the symbolic expressions used as plot inputs.
- [Hamiltonian dynamics worksheet](../worksheets/hamiltonian_dynamics.ipynb) for plotting inside a longer mathematical worksheet.